# 03 RAG 完整流程

检索 + 生成：从知识库找到相关文档，让 LLM 基于文档回答问题。

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

from src.embed import embed
from src.client import get_client

## 知识库

In [ ]:
docs = [
    "DeepSeek API 支持流式输出，只需在调用时设置 stream=True 参数，然后迭代接收每个 chunk。",
    "RAG（检索增强生成）通过先从外部知识库检索相关文档，再将文档与问题一起送入 LLM 来提升回答质量。",
    "Python 是一门简洁易读的编程语言，广泛用于数据分析、Web 开发和人工智能领域。",
    "北京今天天气晴朗，气温 25°C，非常适合户外活动如爬山和骑行。",
    "意大利菜以番茄、橄榄油和新鲜香草为基础，披萨和意大利面是最具代表性的美食。",
    "Transformer 架构通过自注意力机制（Self-Attention）捕捉序列中任意两个位置之间的依赖关系，是 LLM 的核心基础。",
    "使用 DeepSeek 的 Chat API 时，可以通过 system prompt 设定助手的行为风格和角色定位。",
    "索引（Index）是向量检索的核心数据结构，存储了所有文档的 embedding 向量以便快速搜索。",
]

doc_vectors = embed(docs)

## RAG：检索 + 生成

In [ ]:
def retrieve(query, top_k=3):
    query_vec = embed([query])
    scores = cosine_similarity(query_vec, doc_vectors)[0]
    ranked = np.argsort(scores)[::-1]
    return [(docs[i], scores[i]) for i in ranked[:top_k]]

def rag(query, top_k=3):
    hits = retrieve(query, top_k)
    context = "\n".join([doc for doc, _ in hits])

    client = get_client()
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": "你是一个基于文档回答问题的助手。只根据提供的文档内容回答，文档中没有的信息就说不知道。"},
            {"role": "user", "content": f"文档：\n{context}\n\n问题：{query}"},
        ],
    )
    answer = response.choices[0].message.content

    print("检索到的文档：")
    for doc, score in hits:
        print(f"  [{score:.3f}] {doc}")
    print(f"\n回答：{answer}")

rag("DeepSeek 支持流式输出吗？")
print("\n" + "="*50 + "\n")
rag("北京今天天气怎么样？")
print("\n" + "="*50 + "\n")
rag("什么是自注意力？")